In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from torchvision.utils import make_grid

# Set random seed for reproducibility
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create directories for saving model checkpoints and generated images
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('generated_images', exist_ok=True)

# Load dataset (MNIST)
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

# Spectral Normalization implementation for discriminator stability
def spectral_norm(module):
    nn.utils.spectral_norm(module)
    return module

# Define the GAN Generator architecture
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super(Generator, self).__init__()
        # Update initial size to match target 28x28 images
        self.init_size = 7  # 7x7 is chosen so that two upscaling steps yield 28x28
        self.l1 = nn.Sequential(
            nn.Linear(z_dim, 128 * self.init_size * self.init_size)
        )

        # Use two ConvTranspose2d layers to upscale from 7x7 -> 14x14 -> 28x28
        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(128),
            # Upsample from 7x7 to 14x14
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            # Upsample from 14x14 to 28x28
            nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh()  # Output pixel values between -1 and 1
        )

    def forward(self, z):
        # Project and reshape noise vector
        out = self.l1(z.view(z.size(0), -1))
        out = out.view(out.size(0), 128, self.init_size, self.init_size)
        # Convolutional transpose blocks
        out = self.conv_blocks(out)
        return out


# Define the GAN Discriminator architecture with reduced capacity
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        # Input: 1x28x28 (MNIST image)
        # Reduced number of filters and one fewer layer compared to original
        self.conv_blocks = nn.Sequential(
            # 1x28x28 -> 32x14x14 (reduced from 64 filters)
            spectral_norm(nn.Conv2d(1, 32, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.3),  # Increased dropout

            # 32x14x14 -> 64x7x7 (reduced from 128 filters)
            spectral_norm(nn.Conv2d(32, 64, 4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.3),  # Increased dropout
        )

        # Fully connected layer for classification
        # We need to calculate the correct input size: 64 filters × 7×7 feature map = 3136
        self.fc = spectral_norm(nn.Linear(64 * 7 * 7, 1))

    def forward(self, x):
        # Add noise to discriminator input for stability
        if self.training:
            x = x + 0.1 * torch.randn_like(x)

        # Convolutional blocks
        x = self.conv_blocks(x)

        # Flatten and feed to fully connected layer
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

# Function to train the GAN with improved stability
def train_gan(generator, discriminator, g_optimizer, d_optimizer, dataloader,
              num_epochs=20, z_dim=100, checkpoint_path=None, save_interval=5):
    """
    Train a GAN model with improved stability techniques

    Parameters:
    -----------
    generator: Generator model
    discriminator: Discriminator model
    g_optimizer: Optimizer for the generator
    d_optimizer: Optimizer for the discriminator
    dataloader: DataLoader for training data
    num_epochs: Number of training epochs
    z_dim: Dimension of the noise vector
    checkpoint_path: Path to save model checkpoints
    save_interval: Interval for saving images and checkpoints
    """
    # Lists to store losses
    g_losses = []
    d_losses = []

    # Label smoothing
    real_label = 0.9  # One-sided label smoothing
    fake_label = 0.0

    # Fixed noise for visualization
    fixed_noise = torch.randn(64, z_dim, 1, 1).to(device)

    # Binary cross entropy loss
    criterion = nn.BCEWithLogitsLoss()

    # Normalize MNIST images between -1 and 1
    transform_norm = transforms.Compose([
        transforms.Normalize(mean=[-0.5/0.5], std=[1/0.5])
    ])

    print("Starting improved GAN training with:")
    print(f"- Discriminator reduced capacity (32 and 64 filters)")
    print(f"- Discriminator input noise (scale 0.1)")
    print(f"- Spectral normalization for stability")
    print(f"- Discriminator training every other batch")
    print(f"- Generator training twice per batch")
    print(f"- Discriminator learning rate: 0.0001 (reduced)")

    for epoch in range(num_epochs):
        for batch_idx, (real_imgs, _) in enumerate(dataloader):
            # Move data to device
            real_imgs = real_imgs.to(device)
            batch_size = real_imgs.size(0)

            # Transform real images to have range [-1, 1]
            real_imgs = transform_norm(real_imgs)

            # -----------------
            # Train Discriminator (less frequently - every other batch)
            # -----------------
            if batch_idx % 2 == 0:
                d_optimizer.zero_grad()

                # Real images
                real_validity = discriminator(real_imgs)
                real_labels = torch.full((batch_size, 1), real_label, device=device)
                d_real_loss = criterion(real_validity, real_labels)

                # Fake images
                z = torch.randn(batch_size, z_dim, 1, 1).to(device)
                fake_imgs = generator(z)
                fake_validity = discriminator(fake_imgs.detach())
                fake_labels = torch.full((batch_size, 1), fake_label, device=device)
                d_fake_loss = criterion(fake_validity, fake_labels)

                # Total discriminator loss
                d_loss = d_real_loss + d_fake_loss
                d_loss.backward()
                d_optimizer.step()
            else:
                d_loss = torch.tensor(0.0)  # Placeholder for logging when we skip D update

            # -----------------
            # Train Generator (twice per batch)
            # -----------------
            for _ in range(2):  # Train generator twice for each discriminator update
                g_optimizer.zero_grad()

                # Generate batch of fake images
                z = torch.randn(batch_size, z_dim, 1, 1).to(device)
                fake_imgs = generator(z)

                # Try to fool the discriminator
                fake_validity = discriminator(fake_imgs)

                # We want discriminator to predict these as real
                g_loss = criterion(fake_validity, torch.full((batch_size, 1), real_label, device=device))

                g_loss.backward()
                g_optimizer.step()

            # Print progress
            if batch_idx % 100 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}] Batch {batch_idx}/{len(dataloader)} "
                      f"D_loss: {d_loss.item():.4f}, G_loss: {g_loss.item():.4f}")

        # Save losses
        g_losses.append(g_loss.item())
        d_losses.append(d_loss.item())

        # Generate and save images
        if (epoch + 1) % save_interval == 0 or epoch == 0:
            with torch.no_grad():
                fake_imgs = generator(fixed_noise)
                img_grid = make_grid(fake_imgs, nrow=8, normalize=True)
                plt.figure(figsize=(10, 10))
                plt.imshow(img_grid.cpu().permute(1, 2, 0).numpy(), cmap='gray')
                plt.axis('off')
                plt.title(f'GAN Generated Images - Epoch {epoch+1}')
                plt.savefig(f'generated_images/improved_gan_epoch_{epoch+1}.png')
                plt.close()

        # Save checkpoint
        if checkpoint_path and (epoch + 1) % save_interval == 0:
            torch.save({
                'epoch': epoch,
                'generator_state_dict': generator.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
                'g_optimizer_state_dict': g_optimizer.state_dict(),
                'd_optimizer_state_dict': d_optimizer.state_dict(),
            }, checkpoint_path)
            print(f"Checkpoint saved at epoch {epoch+1}")

    # Plot losses
    plt.figure(figsize=(10, 5))
    plt.plot(g_losses, label='Generator Loss')
    plt.plot(d_losses, label='Discriminator Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Improved GAN Training Losses')
    plt.legend()
    plt.savefig('improved_gan_losses.png')
    plt.close()

    return generator, discriminator

# Function to generate and visualize samples from trained generator
def generate_samples(generator, z_dim=100, num_samples=100):
    """
    Generate samples from trained generator

    Parameters:
    -----------
    generator: Trained generator model
    z_dim: Dimension of noise vector
    num_samples: Number of samples to generate
    """
    generator.eval()

    # Generate random noise
    with torch.no_grad():
        # Create random noise vectors
        z = torch.randn(num_samples, z_dim, 1, 1).to(device)

        # Generate images
        generated_images = generator(z)

        # Create a grid of images
        img_grid = make_grid(generated_images, nrow=int(np.sqrt(num_samples)), normalize=True)

        # Display and save
        plt.figure(figsize=(12, 12))
        plt.imshow(img_grid.cpu().permute(1, 2, 0).numpy(), cmap='gray')
        plt.axis('off')
        plt.title('Improved GAN Generated Samples')
        plt.savefig('improved_gan_generated_samples.png')
        plt.close()

        print(f"Generated {num_samples} samples and saved to 'improved_gan_generated_samples.png'")

# Interpolation between points in latent space to show smooth transitions
def latent_space_interpolation(generator, z_dim=100, steps=10):
    """
    Interpolate between points in latent space

    Parameters:
    -----------
    generator: Trained generator model
    z_dim: Dimension of noise vector
    steps: Number of interpolation steps
    """
    generator.eval()

    # Generate two random points in latent space
    z_start = torch.randn(1, z_dim, 1, 1).to(device)
    z_end = torch.randn(1, z_dim, 1, 1).to(device)

    # Create interpolation steps
    alphas = torch.linspace(0, 1, steps)
    z_interpolated = torch.zeros(steps, z_dim, 1, 1).to(device)

    # Linear interpolation between z_start and z_end
    for i, alpha in enumerate(alphas):
        z_interpolated[i] = z_start * (1 - alpha) + z_end * alpha

    # Generate images from interpolated points
    with torch.no_grad():
        interpolated_images = generator(z_interpolated)

        # Create a grid of interpolated images
        img_grid = make_grid(interpolated_images, nrow=steps, normalize=True)

        # Display and save
        plt.figure(figsize=(16, 4))
        plt.imshow(img_grid.cpu().permute(1, 2, 0).numpy(), cmap='gray')
        plt.axis('off')
        plt.title('Improved GAN Latent Space Interpolation')
        plt.savefig('improved_gan_latent_interpolation.png')
        plt.close()

        print("Latent space interpolation saved to 'improved_gan_latent_interpolation.png'")

# Main execution
if __name__ == "__main__":
    # Define hyperparameters
    z_dim = 100  # Noise dimension
    num_epochs = 30

    # Initialize models
    generator = Generator(z_dim=z_dim).to(device)
    discriminator = Discriminator().to(device)

    # Initialize optimizers with different learning rates
    g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
    d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))  # Reduced learning rate

    # Train the GAN
    generator, discriminator = train_gan(
        generator, discriminator, g_optimizer, d_optimizer, train_loader,
        num_epochs=num_epochs, z_dim=z_dim, checkpoint_path='checkpoints/improved_gan.pt',
        save_interval=5
    )

    # Generate samples
    generate_samples(generator, z_dim=z_dim, num_samples=100)

    # Generate latent space interpolation
    latent_space_interpolation(generator, z_dim=z_dim, steps=10)

    print("Improved GAN training and evaluation completed!")